# Backend (teen macilla ja linux pc:llä)

In [1]:
import os
import platform
import pandas as pd

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras

# Apple Silicon Macilla TensorFlow käyttää CPU:ta nopeempaa ja tarkempaa mallia varten
if platform.system() == "Darwin" and platform.machine() == "arm64":
    tf.config.set_visible_devices([], "GPU")

print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("TensorFlow:", tf.__version__)
print("CPU:", tf.config.list_physical_devices("CPU"))
print("GPU:", tf.config.list_physical_devices("GPU"))

Keras: 3.15.1
Backend: tensorflow
TensorFlow: 2.18.1
CPU: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Jako validaatioon, treeniin ja testiin

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}")
print(f"y_test: {y_test.shape}")

x_train: (50000, 32, 32, 3)
y_train: (50000, 1)
x_test: (10000, 32, 32, 3)
y_test: (10000, 1)


In [5]:
print(x_train.shape[0] * 0.8)

40000.0


In [12]:
train_dataset_x = x_train[:40000]
train_dataset_y = y_train[:40000]
validation_dataset_x = x_train[40000:]
validation_dataset_y = y_train[40000:]
test_dataset = [x_test, y_test]


print(f"Treeni koko:\n  x: {train_dataset_x.shape}\n  y: {train_dataset_y.shape}")
print(f"Validaatio koko:\n  x {validation_dataset_x.shape}\n  y: {validation_dataset_y.shape}")
print(f"Testi:\n  x: {test_dataset[0].shape}\n  y: {test_dataset[1].shape}")

Treeni koko:
  x: (40000, 32, 32, 3)
  y: (40000, 1)
Validaatio koko:
  x (10000, 32, 32, 3)
  y: (10000, 1)
Testi:
  x: (10000, 32, 32, 3)
  y: (10000, 1)


## VGG16 lataus

In [15]:
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input

# Ladataan esikoulutettu VGG16
vgg16 = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)

# Jäädytetään painokertoimet
conv_base.trainable = False

# Tarkastellaan mallia
conv_base.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

## Datan prosessointi VGG16 sopivaksi

In [16]:
import numpy as np

train_dataset_x_processed = preprocess_input(train_dataset.astype("float32"))
validation_dataset_x_processed = preprocess_input(validation_dataset.astype("float32"))

## Piirteiden määritys käyttäen VGG16

In [18]:
# Koulutusdatan piirteet
train_features = conv_base.predict(
    train_dataset_x_processed,
    batch_size=32,
    verbose=1
)

# Validointidatan piirteet
val_features = conv_base.predict(
    validation_dataset_x_processed,
    batch_size=32,
    verbose=1
)

print("Treenin ominaisuudet:", train_features.shape)
print("Validoinnin omainisuudet", val_features.shape)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 60s 48ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step
Treenin ominaisuudet: (40000, 1, 1, 512)
Validoinnin omainisuudet (10000, 1, 1, 512)
